In [2]:
import torch
import sys
sys.path.append("src")
from game.utils import Constants, Translator
from game.state import State
from game.game import Game


In [3]:
torch.save(Constants.initial_board(), "tests/fixtures/boards/init_board.pt")

In [4]:
state = State.initial_state()
torch.save([state.get_player_mask(False), state.get_player_mask(True)], "tests/fixtures/game_fixtures/player_masks.pt")
state.save("tests/fixtures/boards/init_board.json")

In [5]:
state.save("temp/test_state_saving.json")

In [6]:
State.from_file("temp/test_state_saving.json").state.shape

torch.Size([6, 4, 8])

In [7]:
from agents.mcts import Tree, Node

In [8]:
game = Game.new_game()
tree = Tree(game, Constants.BATCH_SIZE, 2)
tree.root.set(game.full_move_tensor(1), 1)

In [9]:
for _ in range(256):
    end_node = tree.root.rollout()
    end_node.set(game.full_move_tensor(1), 1)
    # print([(i, node) for i, node in enumerate(tree.root.children) if node is not None])
    root = end_node.backpropagate(end_node._parent_child_index)
    # print(root.visit_count)


In [10]:
tree.to_dot("temp/tree.dot")

In [11]:
import timeit
from typing import cast

stmt = "a = root.mean_value if isinstance(root, Node) else 0"  # or use "type(root) is Node"
# stmt = "a = cast(Node, root).mean_value"  # or use "type(root) is Node"
reps = 1_000_000

duration = timeit.timeit(stmt, globals=globals(), number=reps)
print(f"{stmt}: {duration:.4f} sec over {reps} runs, avg {duration/reps*1e6:.2f} μs/run")

a = root.mean_value if isinstance(root, Node) else 0: 0.0639 sec over 1000000 runs, avg 0.06 μs/run
